# Modeling

### Train/Test Split

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv('heart_2022_PREPROCESSED.csv')

In [3]:
# Target
y = df["HadHeartAttack"]

# Features
X = df.drop(columns=["HadHeartAttack"])

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (246013, 39)
y shape: (246013,)

Target distribution:
HadHeartAttack
0    232578
1     13435
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (196810, 39)
Testing set: (49203, 39)

Training target distribution:
HadHeartAttack
0    0.945389
1    0.054611
Name: proportion, dtype: float64

Testing target distribution:
HadHeartAttack
0    0.94539
1    0.05461
Name: proportion, dtype: float64


### The following preprocessing step does the following: 1) Fill missing numerical values with the median; 2) Standardize numerical variables; 3) Fill missing categorical values with the most common category; 4) One-hot encode the remaining categorical variables

In [7]:
categorical_columns = [
    "Sex",
    "ECigaretteUsage",
    "RaceEthnicityCategory",
    "TetanusLast10Tdap",
    "CovidPos"
]

print("Categorical variables to one-hot encode:")
print(categorical_columns)

Categorical variables to one-hot encode:
['Sex', 'ECigaretteUsage', 'RaceEthnicityCategory', 'TetanusLast10Tdap', 'CovidPos']


In [8]:
numeric_columns = [
    "GeneralHealth",
    "AgeCategory",
    "SmokerStatus",
    "LastCheckupTime",
    "RemovedTeeth",
    "PhysicalHealthDays",
    "MentalHealthDays",
    "SleepHours",
    "HeightInMeters",
    "WeightInKilograms",
    "BMI"
]

# Binary variables converted to 0/1
numeric_columns += [
    "PhysicalActivities",
    "HadAngina",
    "HadStroke",
    "HadAsthma",
    "HadSkinCancer",
    "HadCOPD",
    "HadDepressiveDisorder",
    "HadKidneyDisease",
    "HadArthritis",
    "DeafOrHardOfHearing",
    "BlindOrVisionDifficulty",
    "DifficultyConcentrating",
    "DifficultyWalking",
    "DifficultyDressingBathing",
    "DifficultyErrands",
    "ChestScan",
    "AlcoholDrinkers",
    "HIVTesting",
    "FluVaxLast12",
    "PneumoVaxEver",
    "HighRiskLastYear"
]

print("Number of numeric/binary features:", len(numeric_columns))
print("Number of categorical features:", len(categorical_columns))

Number of numeric/binary features: 32
Number of categorical features: 5


In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

### Prevent information leaking

In [10]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (196810, 50)
Processed testing shape: (49203, 50)


In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))

Number of final features: 50


## The following step will save the processed datasets so you can use them in different Juptyer notebook files

In [13]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed training data:")
display(X_train_processed.head())

print("Processed testing data:")
display(X_test_processed.head())

Processed training data:


,numeric__GeneralHealth,numeric__AgeCategory,numeric__SmokerStatus,numeric__LastCheckupTime,numeric__RemovedTeeth,numeric__PhysicalHealthDays,numeric__MentalHealthDays,numeric__SleepHours,numeric__HeightInMeters,numeric__WeightInKilograms,...,"categorical__RaceEthnicityCategory_Multiracial, Non-Hispanic","categorical__RaceEthnicityCategory_Other race only, Non-Hispanic","categorical__RaceEthnicityCategory_White only, Non-Hispanic","categorical__TetanusLast10Tdap_No, did not receive any tetanus shot in the past 10 years","categorical__TetanusLast10Tdap_Yes, received Tdap","categorical__TetanusLast10Tdap_Yes, received tetanus shot but not sure what type","categorical__TetanusLast10Tdap_Yes, received tetanus shot, but not Tdap",categorical__CovidPos_No,categorical__CovidPos_Tested positive using home test without a health professional,categorical__CovidPos_Yes
84379,0.493686,-1.649695,-0.667691,0.437130,-0.781918,-0.489987,-0.267587,0.679918,-1.736122,0.545808,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
97062,-0.474664,-1.068867,-0.667691,0.437130,0.362768,-0.014386,-0.020748,-0.014732,0.233397,-1.155419,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
148098,0.493686,0.673619,0.428492,0.437130,0.362768,-0.489987,-0.514426,-0.014732,-0.516896,-0.091918,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
141754,-0.474664,-0.197624,-0.667691,-2.138277,-0.781918,-0.489987,-0.514426,0.679918,0.420970,0.290718,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
239240,-0.474664,0.092791,0.428492,-2.138277,0.362768,-0.489987,-0.514426,-0.709383,-0.516896,-1.261863,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


Processed testing data:


,numeric__GeneralHealth,numeric__AgeCategory,numeric__SmokerStatus,numeric__LastCheckupTime,numeric__RemovedTeeth,numeric__PhysicalHealthDays,numeric__MentalHealthDays,numeric__SleepHours,numeric__HeightInMeters,numeric__WeightInKilograms,...,"categorical__RaceEthnicityCategory_Multiracial, Non-Hispanic","categorical__RaceEthnicityCategory_Other race only, Non-Hispanic","categorical__RaceEthnicityCategory_White only, Non-Hispanic","categorical__TetanusLast10Tdap_No, did not receive any tetanus shot in the past 10 years","categorical__TetanusLast10Tdap_Yes, received Tdap","categorical__TetanusLast10Tdap_Yes, received tetanus shot but not sure what type","categorical__TetanusLast10Tdap_Yes, received tetanus shot, but not Tdap",categorical__CovidPos_No,categorical__CovidPos_Tested positive using home test without a health professional,categorical__CovidPos_Yes
69281,0.493686,1.428697,-0.667691,0.43713,2.652141,-0.489987,-0.020748,-0.709383,-0.047963,-1.155419,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
9339,-0.474664,0.383205,0.428492,0.43713,-0.781918,-0.489987,-0.514426,-0.014732,0.420970,-0.304806,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
42144,-0.474664,-0.197624,-0.667691,0.43713,-0.781918,-0.489987,-0.514426,-2.098683,-1.267189,1.311548,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
129323,-0.474664,0.092791,-0.667691,0.43713,0.362768,-0.252187,-0.514426,0.679918,1.358836,1.822197,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
143266,-1.443014,0.383205,2.620858,0.43713,1.507455,1.293517,1.953963,-0.709383,-0.516896,-1.049444,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
# Final check of the shapes of the processed data
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)
# Same as before, good

Training shape: (196810, 50)
Testing shape: (49203, 50)


In [16]:
# Save processed feature matrices
X_train_processed.to_csv("X_train_processed.csv", index=False)
X_test_processed.to_csv("X_test_processed.csv", index=False)

# Save target variables
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)